# L3: Multi-agent Customer Support Automation
In this lesson, you will learn about the six key elements which help make Agents perform even better:

* Role Playing
* Focus
* Tools
* Cooperation
* Guardrails
* Memory

In [11]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [12]:
from crewai import Agent, Task, Crew

In [13]:
import os
from utils import get_openai_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'

# Role Playing, Focus and Cooperation

In [14]:
support_agent = Agent(
    role="Representante Sênior de Suporte",
    goal="Ser o representante de suporte mais amigável e útil "
          "da sua equipe",
    backstory=(
        "Você trabalha na crewAI (https://crewai.com) e "
        "agora está fornecendo suporte para {customer}, um cliente super importante "
        "para sua empresa."
        "Você precisa garantir que forneça o melhor suporte!"
        "Certifique-se de fornecer respostas completas, "
        "sem fazer suposições."
    ),
    allow_delegation=False,
    verbose=True
)


LLM value is None


* Ao não definir allow_delegation=False, allow_delegation assume seu valor padrão como True.
* Isso significa que o agente pode delegar seu trabalho a outro agente mais adequado para realizar uma tarefa específica.

In [15]:
support_quality_assurance_agent = Agent(
    role="Especialista em Garantia de Qualidade de Suporte",
    goal="Obter reconhecimento por fornecer a "
          "melhor garantia de qualidade de suporte na sua equipe",
    backstory=(
        "Você trabalha na crewAI (https://crewai.com) e "
        "agora está trabalhando com sua equipe "
        "em uma solicitação de {customer}, garantindo que "
        "o representante de suporte esteja "
        "fornecendo o melhor suporte possível.\n"
        "Você precisa garantir que o representante de suporte "
        "esteja fornecendo respostas completas, "
        "sem fazer suposições."
    ),
    verbose=True
)

LLM value is None


* Role Playing: Ambos os agentes receberam um papel, um objetivo e uma história de fundo.
* Foco: Ambos os agentes foram solicitados a entrar no personagem dos papéis que estão desempenhando.
* Cooperação: o Agente de Garantia de Qualidade de Suporte pode delegar o trabalho ao Agente de Suporte, permitindo que esses agentes trabalhem juntos.

# Tools, Guardrails and Memory

In [16]:
from crewai_tools import SerperDevTool, \
                         ScrapeWebsiteTool, \
                         WebsiteSearchTool

Possíveis ferramentas personalizadas
* Carregar dados do cliente
* Acesse conversas anteriores
* Carregar dados de um CRM
* Verificando relatórios de bugs existentes
* Verificando solicitações de recursos existentes
* Verificando tickets em andamento
* ... e mais

* Algumas maneiras de usar as ferramentas CrewAI.
  - search_tool = SerperDevTool()
  - scrape_tool = ScrapeWebsiteTool()
* Instancie uma ferramenta de raspagem de documentos.
* A ferramenta irá copiar uma página (apenas 1 URL) da documentação do CrewAI.

In [17]:
docs_scrape_tool = ScrapeWebsiteTool(
    website_url="https://docs.crewai.com/how-to/Creating-a-Crew-and-kick-it-off/"
)

Diferentes maneiras de fornecer ferramentas aos agentes
* Nível do Agente: O Agente pode usar a(s) Ferramenta(s) em qualquer Tarefa que executar.
* Nível de Tarefa: O Agente só usará a(s) Ferramenta(s) ao executar aquela Tarefa específica.

Observação: as ferramentas de tarefas substituem as ferramentas do agente.

# Creating Tasks
Você está passando a ferramenta no nível de tarefa.

In [18]:
inquiry_resolution = Task(
    description=(
        "{customer} acabou de entrar em contato com uma solicitação super importante:\n"
        "{inquiry}\n\n"
        "{person} de {customer} é a pessoa que entrou em contato. "
        "Certifique-se de usar tudo o que você sabe "
        "para fornecer o melhor suporte possível."
        "Você deve se esforçar para fornecer uma resposta completa "
        "e precisa à solicitação do cliente."
    ),
    expected_output=(
        "Uma resposta detalhada e informativa à "
        "solicitação do cliente que aborde "
        "todos os aspectos da pergunta dele.\n"
        "A resposta deve incluir referências "
        "a tudo o que foi utilizado para encontrar a resposta, "
        "incluindo dados ou soluções externas. "
        "Garanta que a resposta seja completa, "
        "sem deixar perguntas sem resposta, e mantenha um tom útil e amigável "
        "ao longo de toda a comunicação."
    ),
    tools=[docs_scrape_tool],
    agent=support_agent,
)

* quality_assurance_review não está usando nenhuma ferramenta(s)
* Aqui o Agente de QA revisará apenas o trabalho do Agente de Suporte

In [19]:
quality_assurance_review = Task(
    description=(
        "Revise a resposta redigida pelo Representante Sênior de Suporte para a solicitação de {customer}. "
        "Garanta que a resposta seja abrangente, precisa e siga os "
        "padrões de alta qualidade esperados para o suporte ao cliente.\n"
        "Verifique se todas as partes da solicitação do cliente "
        "foram abordadas de forma completa, "
        "com um tom útil e amigável.\n"
        "Verifique as referências e fontes usadas para "
        "encontrar as informações, "
        "garantindo que a resposta seja bem fundamentada e "
        "não deixe nenhuma pergunta sem resposta."
    ),
    expected_output=(
        "Uma resposta final, detalhada e informativa "
        "pronta para ser enviada ao cliente.\n"
        "Esta resposta deve abordar completamente a "
        "solicitação do cliente, incorporando todo "
        "o feedback e melhorias relevantes.\n"
        "Não seja muito formal, somos uma empresa descontraída e legal, "
        "mas mantenha um tom profissional e amigável ao longo de toda a comunicação."
    ),
    agent=support_quality_assurance_agent,
)

# Creating the Crew

Memory
* Definir memory=True ao reunir a tripulação habilita Memory.

In [22]:
crew = Crew(
  agents=[support_agent, support_quality_assurance_agent],
  tasks=[inquiry_resolution, quality_assurance_review],
  verbose=True,
  memory=True
)

# Running the Crew
Observação: os LLMs podem fornecer saídas diferentes para a mesma entrada, portanto, o que você obtém pode ser diferente do que vê no vídeo.

Guardrails
* Ao executar a execução abaixo, é possível perceber que os agentes e as respostas estão dentro do escopo do que esperamos deles.

In [ ]:
inputs = {
    "customer": "DeepLearningAI",
    "person": "Andrew Ng",
    "inquiry": "Preciso de ajuda para configurar um Crew "
               "e iniciá-lo, especificamente "
               "como posso adicionar memória ao meu crew? "
               "Você pode fornecer orientações?"
}
result = crew.kickoff(inputs=inputs)

# Agent: Representante Sênior de Suporte
## Task: DeepLearningAI acabou de entrar em contato com uma solicitação super importante:
Preciso de ajuda para configurar um Crew e iniciá-lo, especificamente como posso adicionar memória ao meu crew? Você pode fornecer orientações?

Andrew Ng de DeepLearningAI é a pessoa que entrou em contato. Certifique-se de usar tudo o que você sabe para fornecer o melhor suporte possível.Você deve se esforçar para fornecer uma resposta completa e precisa à solicitação do cliente.


# Agent: Representante Sênior de Suporte
## Thought: Para fornecer a melhor ajuda ao cliente, preciso garantir que entenda completamente como adicionar memória ao Crew dele. Vou começar lendo o conteúdo relevante em nosso site para me certificar de que tenho as informações mais precisas e atualizadas.
## Using tool: Read website content
## Tool Input: 
"{}"
## Tool Output: 
Introduction - CrewAI CrewAI home page Search CrewAI docs crewAIInc / crewAI crewAIInc / crewAI Search... 

In [ ]:
from IPython.display import Markdown

texto_markdown = result.tasks_output[0].raw  
Markdown(texto_markdown)

Oi Andrew,

Obrigado por entrar em contato com a crewAI. Infelizmente, as informações específicas sobre como adicionar memória a um Crew não estão disponíveis na documentação online.

No entanto, posso compartilhar com você algumas informações gerais sobre como configurar um Crew e iniciá-lo. A crewAI permite que você crie equipes de agentes de IA onde cada agente tem funções, ferramentas e objetivos específicos, trabalhando juntos para realizar tarefas complexas.

Aqui estão os componentes principais de uma configuração de Crew:

1. Crew: A organização de alto nível que gerencia as equipes de agentes de IA, supervisiona os fluxos de trabalho, garante a colaboração e entrega os resultados.
2. AI Agents: Membros da equipe especializados que têm funções específicas (pesquisador, escritor), usam ferramentas designadas, podem delegar tarefas e tomam decisões autônomas.
3. Process: Sistema de gerenciamento de fluxo de trabalho que define padrões de colaboração, controla atribuições de tarefas, gerencia interações e garante execução eficiente.
4. Tasks: Tarefas individuais que têm objetivos claros, usam ferramentas específicas, alimentam o processo maior e produzem resultados acionáveis.

Quanto à sua pergunta sobre adicionar memória a um Crew, acredito que você pode estar se referindo a adicionar mais recursos para lidar com tarefas e atividades. Isso geralmente envolve aprimorar as especificações do seu hardware ou software. Dito isso, sem mais detalhes, é difícil fornecer uma resposta definitiva. 

Espero que essas informações sejam úteis, mesmo sem a orientação específica sobre a adição de memória. Se você tiver mais perguntas ou puder fornecer mais detalhes sobre o que você está tentando realizar, ficarei feliz em ajudar mais.

Atenciosamente,
[Seu Nome]
Representante Sênior de Suporte, crewAI.